# Fase 1: Caricamento e Analisi delle Feature
In questa sezione carichiamo il dataset `Base.csv`, effettuiamo il bilanciamento delle classi tramite **downsampling** della classe maggioritaria (transazioni legali) e utilizziamo un modello di **Random Forest** per identificare le 4 feature che hanno il maggior impatto sulla predizione delle frodi. 

Le feature selezionate verranno poi scalate in un range $[0, \pi]$ per essere compatibili con le rotazioni dei gate quantistici.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Caricamento Dataset
dataset_path = 'dataset/Base.csv'
data = pd.read_csv(dataset_path)

# 2. Bilanciamento del Dataset (Downsampling)
# Separiamo le classi
fraud = data[data['fraud_bool'] == 1]
legal = data[data['fraud_bool'] == 0]

# Riduciamo la classe 'legal' per pareggiare il numero di 'fraud'
legal_downsampled = resample(legal, 
                             replace=False, 
                             n_samples=len(fraud), 
                             random_state=42)

# Uniamo e mescoliamo
balanced_data = pd.concat([fraud, legal_downsampled])

# 3. Encoding delle variabili categoriche
le = LabelEncoder()
cat_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']
for col in cat_cols:
    balanced_data[col] = le.fit_transform(balanced_data[col])

# Preparazione X e y
X = balanced_data.drop('fraud_bool', axis=1)
y = balanced_data['fraud_bool']

# 4. Feature Selection con Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Estrazione delle top 4 feature
importances = pd.Series(rf.feature_importances_, index=X.columns).nlargest(4)
top_features = importances.index.tolist()
X_selected = X[top_features].values

print('Top 4 feature selezionate:')
print(importances)

# 5. Scaling per il Calcolo Quantistico (Range 0 - PI)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_final = scaler.fit_transform(X_selected)

print(f'\nDataset pronto. Forma: {X_final.shape}')

# Fase 2: Costruzione della Feature Map Quantistica
Per mappare i dati nel piano di Hilbert, utilizziamo un circuito custom composto da:
1. **Layer di rotazione (Ry)**: codifica i valori delle feature come angoli di rotazione.
2. **Layer di Entanglement**: utilizza gate CNOT e Rz per creare interazioni tra coppie di feature ($x_i \cdot x_j$).
Questo processo viene ripetuto per 2 "reps" (ripetizioni) per aumentare la profondità e la capacità espressiva del circuito.

In [ ]:
from qiskit.circuit import QuantumCircuit, ParameterVector

def build_custom_feature_map(num_qubits, reps=2):
    """
    Crea un circuito che codifica i dati classici in stati quantistici.
    """
    params = ParameterVector('x', length=num_qubits)
    qc = QuantumCircuit(num_qubits)
    
    for r in range(reps):
        # 1. Applicazione gate Hadamard e Ry per ogni feature
        for i in range(num_qubits):
            qc.h(i)
            qc.ry(params[i], i)
        
        # 2. Entanglement completo: interazione tra ogni coppia di feature
        # Implementa una logica simile alla ZZFeatureMap ma customizzata
        for i in range(num_qubits):
            for j in range(i + 1, num_qubits):
                qc.cx(i, j)
                qc.rz(params[i] * params[j], j)
                qc.cx(i, j)
        
        qc.barrier() # Separatore visivo per i layer
    
    return qc

# Inizializzazione circuito a 4 qubit
num_qubits = 4
feature_map = build_custom_feature_map(num_qubits=num_qubits, reps=2)

print(f"Feature Map creata con {feature_map.num_parameters} parametri.")
# feature_map.draw(output='mpl') # Decommenta per visualizzare il grafico

# Fase 3: Definizione dell'Ansatz (Circuito Variazionale)
L'Ansatz è la parte del circuito con parametri "addestrabili" ($\theta$). 
Abbiamo progettato un circuito con **Entanglement Circolare**: ogni qubit è connesso al successivo, e l'ultimo si riconnette al primo. Questo garantisce che le informazioni fluiscano attraverso tutti i qubit in modo efficiente durante l'ottimizzazione.

In [ ]:
from qiskit.circuit import QuantumCircuit, ParameterVector

def build_custom_ansatz(num_qubits, reps=3):
    """
    Costruisce l'architettura del classificatore:
    - Layer di rotazioni Ry parametriche
    - Entanglement circolare tramite gate CX
    """
    # Ogni layer ha 'num_qubits' parametri + un layer finale di Ry
    total_params = num_qubits * reps + num_qubits
    params = ParameterVector('θ', length=total_params)
    
    qc = QuantumCircuit(num_qubits)
    param_idx = 0
    
    for layer in range(reps):
        # Layer di rotazione Ry addestrabile
        for qubit in range(num_qubits):
            qc.ry(params[param_idx], qubit)
            param_idx += 1
        
        # Entanglement Circolare: CX(0,1), CX(1,2), CX(2,3), CX(3,0)
        for qubit in range(num_qubits - 1):
            qc.cx(qubit, qubit + 1)
        qc.cx(num_qubits - 1, 0)
        
        qc.barrier()
    
    # Layer finale di rotazione per stabilizzare l'output
    for qubit in range(num_qubits):
        qc.ry(params[param_idx], qubit)
        param_idx += 1
    
    return qc

# Creazione dell'ansatz con 3 ripetizioni
ansatz = build_custom_ansatz(num_qubits=4, reps=3)
print(f"Ansatz configurato con {ansatz.num_parameters} parametri addestrabili.")

# Fase 4: Configurazione del Classificatore (VQC)
In questa sezione:
1. Suddividiamo i dati in **Training Set** (80%) e **Test Set** (20%).
2. Utilizziamo l'ottimizzatore **SPSA** (Simultaneous Perturbation Stochastic Approximation), ideale per circuiti quantistici rumorosi o simulati, poiché richiede solo due valutazioni della funzione obiettivo per iterazione.
3. Impostiamo un **Callback** per monitorare la convergenza del modello durante il training.

In [ ]:
import time
from sklearn.model_selection import train_test_split
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import SPSA
from qiskit_machine_learning.algorithms import VQC
from IPython.display import clear_output
import matplotlib.pyplot as plt

# 1. Suddivisione del dataset (stratificata per mantenere il bilanciamento)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Setup del sistema di visualizzazione del training
objective_func_vals = []

def callback_graph(nfev, x, fx, dx, accept):
    """
    Funzione chiamata ad ogni iterazione dell'ottimizzatore per aggiornare il grafico.
    """
    clear_output(wait=True)
    objective_func_vals.append(fx)
    plt.figure(figsize=(8, 4))
    plt.title('Training Progress: Loss Function')
    plt.xlabel('Iterazione')
    plt.ylabel('Loss')
    plt.plot(range(len(objective_func_vals)), objective_func_vals, color='darkorange')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 3. Inizializzazione Ottimizzatore e Sampler
# Usiamo 400 iterazioni di SPSA
sampler = StatevectorSampler()
optimizer = SPSA(maxiter=400, callback=callback_graph)

# 4. Assemblaggio del VQC
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    sampler=sampler,
)

print(f"VQC pronto per il training. Campioni di training: {len(X_train)}")

# Fase 5: Addestramento e Analisi delle Performance
Per ottimizzare i tempi di calcolo, effettuiamo il training su un subset di 1500 campioni.
Dopo il training:
1. Valutiamo l'**AUC (Area Under the Curve)** per capire la capacità discriminante del modello.
2. Calcoliamo la **Soglia Ottimale** tramite l'indice di Youden (punto della ROC più vicino all'angolo ideale).
3. Visualizziamo la **Matrice di Confusione** per vedere quante frodi sono state correttamente identificate.

In [ ]:
import time
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

# 1. Training su un subset (per efficienza computazionale)
subset_size = 1500
X_train_small = X_train[:subset_size]
y_train_small = y_train.values[:subset_size]

print(f"Inizio training su {subset_size} campioni...")
start_time = time.time()
vqc.fit(X_train_small, y_train_small)
end_time = time.time()
print(f"Training completato in {(end_time - start_time)/60:.2f} minuti.")

# 2. Predizione e calcolo probabilità
y_proba = vqc.predict_proba(X_test)
y_scores = y_proba[:, 1]  # Probabilità della classe 'Frode'

# 3. Analisi ROC e ricerca soglia ottimale
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

# Indice di Youden: massimizza (True Positive Rate - False Positive Rate)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

# Predizione con la nuova soglia
y_pred_opt = (y_scores >= optimal_threshold).astype(int)

# 4. Visualizzazione Risultati
print(f"\nAUC Score: {auc:.3f}")
print(f"Soglia ottimale: {optimal_threshold:.3f}")
print("\nClassification Report (Soglia Ottimizzata):")
print(classification_report(y_test, y_pred_opt, target_names=['Legale', 'Frode']))

# Plot: ROC Curve + Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot ROC
axes[0].plot(fpr, tpr, label=f'ROC curve (area = {auc:.2f})', color='blue')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', label='Soglia Ottimale')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Receiver Operating Characteristic (ROC)')
axes[0].legend()

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred_opt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legale', 'Frode'])
disp.plot(cmap='Blues', ax=axes[1])
axes[1].set_title('Matrice di Confusione')

plt.tight_layout()
plt.show()